# Cats vs Dogs: MLflow + MinIO tracked training

This notebook is the experiment interface. Reusable implementation lives in
`cats_dogs_pipeline.py`; the notebook configures a run group, invokes the pipeline, records
the standard experiments, and can optionally launch history-aware auto-tuning.

The MLflow Tracking Server proxies all Artifact writes to MinIO. No MinIO credential belongs in
this notebook or in the client process.


## 1. Load the reusable pipeline

Restart the kernel after changing `cats_dogs_pipeline.py` so the notebook imports a clean module.


In [ ]:
import sys
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display

MODULE_DIR = Path.cwd()
if not (MODULE_DIR / "cats_dogs_pipeline.py").is_file():
    MODULE_DIR = Path(
        "/data/ai/chenzhangyue/code/galatea/train-model/cats-and-dogs"
    )
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from cats_dogs_pipeline import (
    PipelineConfig,
    build_model,
    compare_run_group,
    create_generators,
    plot_confusion_matrix,
    plot_dataset_distribution,
    plot_gradcam_examples,
    plot_image_batch,
    plot_prediction_examples,
    plot_training_history,
    preflight_tracking,
    prepare_dataset,
    run_tracked_training,
)
from cats_dogs_tuner import TuningConfig, run_auto_tuning

def make_epoch_progress(label):
    """Return a notebook display that updates once per completed epoch."""
    progress = display(
        HTML(f"<b>{label}</b> waiting for the first epoch..."),
        display_id=True,
    )

    def update(metrics):
        epoch = int(metrics["epoch"])
        total = int(metrics["epochs_requested"])
        percent = min(100, int(100 * epoch / max(1, total)))
        rendered = HTML(
            f"""<div style='min-width:420px'>
            <b>{label}</b> - epoch {epoch}/{total} ({percent}%)<br>
            <progress value='{epoch}' max='{total}' style='width:100%'></progress>
            train loss {metrics['train_loss']:.4f}, train acc {metrics['train_accuracy']:.3f} | 
            val loss {metrics['val_loss']:.4f}, val acc {metrics['val_accuracy']:.3f}
            </div>"""
        )
        if progress is None:
            display(rendered)
        else:
            progress.update(rendered)

    return update


## 2. Configure tracking and fail closed

Change `EPOCHS` and the optional tuning parameters in the next cell before running training.
The cell is tagged `parameters` for automation tools such as Papermill.

The preflight verifies MLflow connectivity and rejects a local Artifact Store. With the repository's
service configuration, `mlflow-artifacts:/` is proxied by MLflow Server to `s3://mlflow-artifacts`
in MinIO.


In [ ]:
# Change these values before running the notebook.
EPOCHS = None  # None uses CATS_DOGS_EPOCHS (default: 1); set 10 for ten epochs.
RUN_AUTO_TUNING = True
TUNER_EPOCHS = 32
TUNER_MAX_TRIALS = 8
TUNER_TARGET_VAL_ACCURACY = 0.95

config = PipelineConfig.from_env()
if EPOCHS is not None:
    config = replace(config, epochs=EPOCHS)
tracking = preflight_tracking(config)

display(pd.Series({
    "tracking_uri": config.tracking_uri,
    "experiment": tracking.experiment_name,
    "experiment_id": tracking.experiment_id,
    "artifact_location": tracking.artifact_location,
    "run_group_id": config.run_group_id,
    "epochs": config.epochs,
    "run_auto_tuning": RUN_AUTO_TUNING,
    "tuner_epochs": TUNER_EPOCHS,
    "tuner_max_trials": TUNER_MAX_TRIALS,
    "tuner_target_val_accuracy": TUNER_TARGET_VAL_ACCURACY,
    "quality_gate_min_accuracy": config.min_test_accuracy,
}))


## 3. Prepare and review the immutable input

Preparation validates every image, calculates SHA-256 checksums, creates deterministic train,
validation, and test splits, and writes the Manifest later logged as MLflow Dataset Inputs.


In [ ]:
dataset = prepare_dataset(config)

display(pd.Series({
    "dataset_version": dataset.dataset_version,
    "source_uri": dataset.source_uri,
    "content_sha256": dataset.content_digest,
    "split_sha256": dataset.split_digest,
    "valid_images": len(dataset.manifest),
    "invalid_images": len(dataset.invalid_files),
}))
display(dataset.split_counts)
display(dataset.invalid_files)
plot_dataset_distribution(dataset)
plt.show()


## 4. Baseline Run


In [ ]:
baseline_data = create_generators(config, dataset, augmented=False)
plot_image_batch(baseline_data.training, n_images=9)
plt.show()


In [ ]:
baseline_model = build_model(config, variant="baseline")
baseline_progress = make_epoch_progress("Baseline")
baseline_result = run_tracked_training(
    config,
    tracking,
    dataset,
    baseline_data,
    baseline_model,
    variant="baseline",
    progress_callback=baseline_progress,
)

display(pd.Series({
    "run_id": baseline_result.run_id,
    "model_uri": baseline_result.model_uri,
    "artifact_uri": baseline_result.artifact_uri,
    "quality_gate_passed": baseline_result.quality_gate_passed,
    **baseline_result.test_metrics,
}))


In [ ]:
plot_training_history(baseline_result.history)
plt.show()
plot_confusion_matrix(baseline_result.confusion_matrix)
plt.show()
plot_prediction_examples(baseline_result, baseline_data.test, n_images=10)
plt.show()


### Baseline explainability

Grad-CAM is a review visualization. The authoritative deployable outputs remain the signed model,
test predictions, metrics, and reports stored with the MLflow Run.


In [ ]:
plot_gradcam_examples(
    baseline_result, baseline_data.test, desired_class=0, n_images=5
)
plt.show()
plot_gradcam_examples(
    baseline_result, baseline_data.test, desired_class=1, n_images=5
)
plt.show()


## 5. Data-augmented Run


In [ ]:
augmented_data = create_generators(config, dataset, augmented=True)
augmented_model = build_model(config, variant="augmented")
augmented_progress = make_epoch_progress("Augmented")
augmented_result = run_tracked_training(
    config,
    tracking,
    dataset,
    augmented_data,
    augmented_model,
    variant="augmented",
    progress_callback=augmented_progress,
)

display(pd.Series({
    "run_id": augmented_result.run_id,
    "model_uri": augmented_result.model_uri,
    "artifact_uri": augmented_result.artifact_uri,
    "quality_gate_passed": augmented_result.quality_gate_passed,
    **augmented_result.test_metrics,
}))


In [ ]:
plot_training_history(augmented_result.history)
plt.show()
plot_confusion_matrix(augmented_result.confusion_matrix)
plt.show()
plot_prediction_examples(augmented_result, augmented_data.test, n_images=10)
plt.show()


## 6. Compare the tracked experiments

The query is scoped to this execution's `run_group_id`; historical runs cannot be mixed into the
decision table. A failed quality gate remains auditable but must not be promoted.


In [ ]:
comparison = compare_run_group(config, tracking)
display(comparison)


## 7. Optional history-aware auto-tuning

Set `RUN_AUTO_TUNING = True` in the parameter cell to run the MLflow history-aware search.
Trial selection uses validation accuracy only. The test split is evaluated once when the best
configuration is retrained as the champion.


In [ ]:
tuning_outcome = None
if RUN_AUTO_TUNING:
    tuning_config = replace(
        TuningConfig.from_env(config.seed),
        epochs_per_trial=TUNER_EPOCHS,
        max_trials=TUNER_MAX_TRIALS,
        target_val_accuracy=TUNER_TARGET_VAL_ACCURACY,
    )
    tuning_outcome = run_auto_tuning(
        config, tuning_config, tracking=tracking, dataset=dataset
    )
    display(pd.Series(asdict(tuning_outcome)))
else:
    print("Auto-tuning skipped; set RUN_AUTO_TUNING = True to enable it.")


## Review contract

Each Run contains Dataset Inputs, content and split digests, parameters, epoch and system metrics,
fixed test metrics, predictions, plots, a best checkpoint, source files, runtime metadata, and a
TensorFlow model with Signature and Input Example. `artifact.roundtrip_verified=true` confirms that
the client downloaded the verification object after MLflow stored it through the Artifact proxy.

Model Registry promotion is intentionally outside this exploratory notebook and should be handled by
an approved quality-gate workflow.
